# Splits y pares siameses sin fuga

Este notebook se ejecuta después de `01_dataset_preparation.ipynb`. El orden es obligatorio: **asignar videos a splits → generar pares dentro de cada split → auditar**. No entrena el modelo.

## Parámetros reproducibles

In [ ]:
RUN_BUILD_SPLITS = True
RUN_BUILD_PAIRS = True
RUN_AUDIT = True
OVERWRITE_PAIRS = True

SEED = 42
TRAIN_PAIRS = 4000
VALIDATION_PAIRS = 500
TEST_PAIRS = 500
SHOW_PAIR_SAMPLES = False
NUM_SAMPLE_PAIRS = 3

## Configuración

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

current_path = Path.cwd()
if (current_path / "src").exists():
    project_root = current_path
elif (current_path.parent / "src").exists():
    project_root = current_path.parent
else:
    raise RuntimeError("No se encontró la raíz del proyecto.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import DATASET_MANIFEST_PATH, PAIRS_DIR

print(f"Proyecto: {project_root}")

In [ ]:
def run_command(command):
    print("$", " ".join(map(str, command)))
    env = os.environ.copy()
    env.update({"PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8"})
    result = subprocess.run(
        command,
        cwd=project_root,
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True,
        env=env,
    )
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr)
        raise RuntimeError(f"El comando falló con código {result.returncode}")

## 1. Asignar imágenes por video

La unidad indivisible es `source_video`. Con cuatro videos por persona, el valor predeterminado 50/25/25 asigna dos videos a train y uno a validation/test. Ningún frame consecutivo puede cruzar splits.

In [ ]:
split_command = [sys.executable, "-m", "src.dataset.build_splits", "--seed", str(SEED)]
if RUN_BUILD_SPLITS:
    run_command(split_command)
else:
    print("Asignación deshabilitada; se usará el split existente en el manifiesto.")

## 2. Generar pares dentro de cada split

In [ ]:
pair_command = [
    sys.executable, "-m", "src.dataset.build_pairs",
    "--seed", str(SEED),
    "--train-pairs", str(TRAIN_PAIRS),
    "--validation-pairs", str(VALIDATION_PAIRS),
    "--test-pairs", str(TEST_PAIRS),
]
if OVERWRITE_PAIRS:
    pair_command.append("--overwrite")
if RUN_BUILD_PAIRS:
    run_command(pair_command)
else:
    print("Generación deshabilitada; se usarán los CSV existentes.")

## 3. Auditoría obligatoria

In [ ]:
if RUN_AUDIT:
    run_command([sys.executable, "-m", "src.dataset.audit_splits"])
else:
    print("ADVERTENCIA: auditoría deshabilitada.")

## Distribución de imágenes, videos y personas

In [ ]:
manifest = pd.read_csv(DATASET_MANIFEST_PATH, keep_default_na=False)
usable = manifest[manifest["status"] == "usable"].copy()

split_summary = usable.groupby("split").agg(
    images=("image_path", "nunique"),
    videos=("source_video", "nunique"),
    people=("person_id", "nunique"),
).reindex(["train", "validation", "test"])
display(split_summary)

person_distribution = usable.pivot_table(
    index="person_id", columns="split", values="image_path", aggfunc="nunique", fill_value=0
).reindex(columns=["train", "validation", "test"])
display(person_distribution)

## Balance de pares

In [ ]:
pair_paths = {
    "train": PAIRS_DIR / "train_pairs.csv",
    "validation": PAIRS_DIR / "val_pairs.csv",
    "test": PAIRS_DIR / "test_pairs.csv",
}
pairs = {split: pd.read_csv(path) for split, path in pair_paths.items()}

pair_summary = pd.DataFrame([
    {
        "split": split,
        "total": len(df),
        "positive": int((df["label"] == 1).sum()),
        "negative": int((df["label"] == 0).sum()),
    }
    for split, df in pairs.items()
]).set_index("split")
display(pair_summary)

## Confirmación independiente de cero solapamiento

In [ ]:
def cross_split_count(sets_by_split):
    owners = {}
    for split, values in sets_by_split.items():
        for value in values:
            owners.setdefault(value, set()).add(split)
    return sum(len(splits) > 1 for splits in owners.values())

image_sets = {s: set(usable.loc[usable["split"] == s, "image_path"]) for s in pairs}
video_sets = {s: set(usable.loc[usable["split"] == s, "source_video"]) for s in pairs}
hash_sets = {s: set(usable.loc[usable["split"] == s, "sha256"]) for s in pairs}
pair_sets = {
    s: {tuple(sorted((row.image_a, row.image_b))) for row in df.itertuples()}
    for s, df in pairs.items()
}

checks = pd.Series({
    "imágenes compartidas": cross_split_count(image_sets),
    "videos compartidos": cross_split_count(video_sets),
    "hashes compartidos": cross_split_count(hash_sets),
    "pares repetidos": cross_split_count(pair_sets),
}, name="count")
display(checks.to_frame())
assert (checks == 0).all(), "Se detectó fuga: revisa la salida de audit_splits."
print("Confirmado: no hay imágenes, videos, hashes ni pares repetidos entre splits.")

## Vista previa opcional

In [ ]:
def show_pairs(df, label, n=3):
    selected = df[df["label"] == label].head(n)
    if selected.empty:
        print("No hay pares para mostrar.")
        return
    fig, axes = plt.subplots(len(selected), 2, figsize=(6, 3 * len(selected)), squeeze=False)
    for row_index, row in enumerate(selected.itertuples()):
        for column, image_path in enumerate([row.image_a, row.image_b]):
            axes[row_index, column].imshow(plt.imread(project_root / image_path))
            axes[row_index, column].axis("off")
        axes[row_index, 0].set_title(f"label={label} | A")
        axes[row_index, 1].set_title("B")
    plt.tight_layout()

if SHOW_PAIR_SAMPLES:
    show_pairs(pairs["train"], 1, NUM_SAMPLE_PAIRS)
    show_pairs(pairs["train"], 0, NUM_SAMPLE_PAIRS)
else:
    print("Vista previa deshabilitada para no guardar salidas pesadas.")

## Resultado esperado

La auditoría debe terminar con `AUDITORÍA APROBADA` y las cuatro comprobaciones independientes deben ser cero. Solo entonces los CSV quedan listos para reentrenar el baseline en una sesión posterior.